# Applicator

I want to apply a theory to a response. In principle, the response should just be a sentence, but it needs to have been read and SpaCy'd first. So for the moment, I'll just assume that we're applying to sentences from the data file. Which, fortunately, is all encoded in the theory object, having been extracted from `bea_2way_data_pickle`.

Want two outputs: one of the results, one of the applications.

The theory itself will be the output from `bea_2way_theory_generator.ipynb/induce_theory`.

Want `apply_theory(theory, original_response_id)`.

Actually, a better plan might be to just have a function which applies the theory to the embedded trial set.

`apply_trial_set(theory)`

In [212]:
THEORY_ROOT_DIR='/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318'

In [213]:
import amati_bea as amati

In [214]:
import pandas as pd

from sklearn.metrics import f1_score, cohen_kappa_score

In [215]:
from operator import itemgetter
import pickle
import datetime
import re
import os
import glob

## Get the data

We'll just be able to pull out the relevant data from the saved theory.

In [216]:
with open(
    "/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260316/theory_0a8e8837-0a58-4e89-96c2-c44ae7281535_2026-03-16_21:09:29.557897.pickle", "rb"
) as fIn:
    theory = pickle.load(fIn)

In [217]:
theory.keys()

dict_keys(['theory', 'original_question_id', 'question_id', 'all_questions_df', 'all_responses_df', 'all_text_df', 'rules'])

Let's find a trial response. First, need to pull out the question for that theory:

In [218]:
theory["question_id"]

'q_05'

And then get the associated responses:

In [219]:
theory["all_responses_df"].query("`question_id`==@theory['question_id']").query(
    "`partition`=='trial'"
).head()

,original_response_id,response_id,question_id,response,score,partition
865,04a4fee4-1dab-48d2-903b-a4159319d652,r_0006,q_05,"Es fällt auf , dass nachdem die Buntbarsche ei...",incorrect,trial
866,af488940-639d-4bec-bab0-15717b25c829,r_0150,q_05,Fisch nummer 4 passt zu den Beobachtung wenig ...,incorrect,trial
867,e38ceb02-4d7a-4558-9582-3e55b0710eba,r_0683,q_05,"In den ersten 6 Monaten , stieg die Anzahl und...",correct,trial
868,f0f4ffc7-bedf-451a-83e7-10a855903d60,r_1527,q_05,bis zum 6. Monat sich die Flecken in beiden Be...,incorrect,trial
869,101e8fe2-9e9c-4216-928a-83d0a33bddbb,r_1723,q_05,In den Becken mit groben Kies und ohne Buntbar...,incorrect,trial


So let's just bag the first of those:

In [220]:
TEST_RESPONSE_ID = (
    theory["all_responses_df"]
    .query("`question_id`==@theory['question_id']")
    .query("`partition`=='trial'")
    .iloc[0, 0]
)

TEST_RESPONSE_ID

'04a4fee4-1dab-48d2-903b-a4159319d652'

Reverse ferret, let's apply to the trial set:

In [221]:
def apply_trial_set(theory):

    question_id=theory["question_id"]
    original_question_id=theory["original_question_id"]

    question_ss = theory['all_questions_df'].set_index("original_question_id").T[
        original_question_id
    ]
    
    trial_responses_df = theory['all_responses_df'].query("`question_id`==@question_id").query(
        "`partition`=='trial'"
    )

    trial_texts_used = [
        question_id,
        question_ss["correct_id"],
        question_ss["incorrect_1_id"],
        question_ss["incorrect_2_id"],
    ] + list(trial_responses_df["response_id"])

    trial_text_df = theory['all_text_df'].query("`id`.isin(@trial_texts_used)")[
        ["id", "i", "lemma"]
    ]

    results_df = amati.evaluate_theory(
        theory['theory'],
        responses_df=trial_responses_df,
        text_df=trial_text_df,
        rulesstring=theory['rules'],
        question_ss=question_ss,
    )

    compare_df = (
        trial_responses_df[["response_id", "score"]]
        .rename({"score": "actual"}, axis="columns")
        .set_index("response_id")
        .assign(
            prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")
        )
    )

    return {"theory": theory, "results": results_df, "compare": compare_df}

In [222]:
def get_rule_applications(theory):

    question_id=theory["question_id"]
    original_question_id=theory["original_question_id"]

    question_ss = theory['all_questions_df'].set_index("original_question_id").T[
        original_question_id
    ]
    
    trial_responses_df = theory['all_responses_df'].query("`question_id`==@question_id").query(
        "`partition`=='trial'"
    )

    trial_texts_used = [
        question_id,
        question_ss["correct_id"],
        question_ss["incorrect_1_id"],
        question_ss["incorrect_2_id"],
    ] + list(trial_responses_df["response_id"])

    trial_text_df = theory['all_text_df'].query("`id`.isin(@trial_texts_used)")[
        ["id", "i", "lemma"]
    ]

    results_dict = amati.apply_theory(
        theory['theory'],
        responses_df=trial_responses_df,
        text_df=trial_text_df,
        rulesstring=theory['rules'],
        question_ss=question_ss,
    )

    """ compare_df = (
        trial_responses_df[["response_id", "score"]]
        .rename({"score": "actual"}, axis="columns")
        .set_index("response_id")
        .assign(
            prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")
        )
    ) """

    return {"theory": theory, "applications": results_dict}

In [223]:
question_id = theory["question_id"]
original_question_id = theory["original_question_id"]

question_ss = (
    theory["all_questions_df"].set_index("original_question_id").T[original_question_id]
)

trial_responses_df = (
    theory["all_responses_df"]
    .query("`question_id`==@question_id")
    .query("`partition`=='trial'")
)

trial_texts_used = [
    question_id,
    question_ss["correct_id"],
    question_ss["incorrect_1_id"],
    question_ss["incorrect_2_id"],
] + list(trial_responses_df["response_id"])

trial_text_df = theory["all_text_df"].query("`id`.isin(@trial_texts_used)")[
    ["id", "i", "lemma"]
]

In [224]:

amati.apply_theory(
    theory["theory"],
    responses_df=trial_responses_df,
    rulesstring=theory["rules"],
    question_ss=question_ss,
    text_df=trial_text_df,
)

{'r_0006': [{'result': 'SATISFIABLE',
   'selected_rule': ('ki_rule1', 'selected_rule(ki_rule1)'),
   'predicted_grade': ('incorrect', 'predicted_grade(incorrect)'),
   'parameters': [(1, '"zwanzig"', 'parameter(1,"zwanzig")')],
   'positive_covered': ['r_0221',
    'r_0358',
    'r_0715',
    'r_0720',
    'r_0880',
    'r_0999',
    'r_1487',
    'r_1554',
    'r_1776',
    'r_1809',
    'r_1820',
    'r_1827',
    'r_1832',
    'r_1853',
    'r_1897',
    'r_1977',
    'r_2117',
    'r_2147',
    'r_2269',
    'r_2946',
    'r_2949',
    'r_3044',
    'r_3162',
    'r_3165',
    'r_3228',
    'r_3243',
    'r_3246',
    'r_3313',
    'r_3361',
    'r_3462',
    'r_3542',
    'r_3602',
    'r_3665',
    'r_3874',
    'r_4311',
    'r_4370',
    'r_4415',
    'r_4501',
    'r_4524',
    'r_4643',
    'r_4679',
    'r_4847',
    'r_4881',
    'r_4997',
    'r_5116',
    'r_5291',
    'r_5536',
    'r_5575',
    'r_5756',
    'r_5774',
    'r_5906',
    'r_5907',
    'r_5924',
    'r_59

In [225]:
o=apply_trial_set(theory)

In [226]:
o['results']

,0,1,2,3,4,5,6,7
response_id,,,,,,,,
r_0006,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0150,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0683,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_1527,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_1723,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_1881,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_3258,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_3716,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_4192,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [227]:
t=get_rule_applications(theory)
t['applications']

{'r_0006': [{'result': 'SATISFIABLE',
   'selected_rule': ('ki_rule1', 'selected_rule(ki_rule1)'),
   'predicted_grade': ('incorrect', 'predicted_grade(incorrect)'),
   'parameters': [(1, '"zwanzig"', 'parameter(1,"zwanzig")')],
   'positive_covered': ['r_0221',
    'r_0358',
    'r_0715',
    'r_0720',
    'r_0880',
    'r_0999',
    'r_1487',
    'r_1554',
    'r_1776',
    'r_1809',
    'r_1820',
    'r_1827',
    'r_1832',
    'r_1853',
    'r_1897',
    'r_1977',
    'r_2117',
    'r_2147',
    'r_2269',
    'r_2946',
    'r_2949',
    'r_3044',
    'r_3162',
    'r_3165',
    'r_3228',
    'r_3243',
    'r_3246',
    'r_3313',
    'r_3361',
    'r_3462',
    'r_3542',
    'r_3602',
    'r_3665',
    'r_3874',
    'r_4311',
    'r_4370',
    'r_4415',
    'r_4501',
    'r_4524',
    'r_4643',
    'r_4679',
    'r_4847',
    'r_4881',
    'r_4997',
    'r_5116',
    'r_5291',
    'r_5536',
    'r_5575',
    'r_5756',
    'r_5774',
    'r_5906',
    'r_5907',
    'r_5924',
    'r_59

## Apply to all theories

In [228]:
outputs_ls=[]

for f in glob.glob(os.path.join(THEORY_ROOT_DIR, '*')):
    print(f)
    with open(f, 'rb') as fIn:
        try:
            theory=pickle.load(fIn)
            outputs_ls.append(apply_trial_set(theory))
        except:
            continue

/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_5f7bd982-606b-45e4-bf39-89db6714a621_2026-03-18_12:52:28.028319.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_35d897f0-c1b4-422c-9a33-43e7da1ffe27_2026-03-18_12:57:35.168998.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_52f6e329-eea6-423b-8d6a-047448d02c3f_2026-03-18_15:45:58.021429.pickle


/var/folders/ql/c79k160s5zjd1mzy_k08xvc40000gq/T/ipykernel_23082/2063899696.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")


/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_c405c4a8-83f2-4ddf-b0cf-941a3356b251_2026-03-18_16:59:35.923009.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_0235896f-1570-418a-a34d-7bc3ea0f82b9_2026-03-18_14:07:56.575271.pickle


/var/folders/ql/c79k160s5zjd1mzy_k08xvc40000gq/T/ipykernel_23082/2063899696.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")


/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_920df344-fe5c-4f48-ae64-a8458e8db40a_2026-03-18_15:24:00.591946.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_c51b89a6-0219-4eee-bb89-b2b5168998f5_2026-03-18_14:12:46.706850.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_0f036c26-4a47-48ba-b913-f27487286ac7_2026-03-18_12:23:14.044820.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_48071573-9e44-4ae9-b08c-2b19b6ee58d6_2026-03-18_12:27:26.893725.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_dad43e50-3167-41d6-996c-5b6e5f200c07_2026-03-18_13:59:21.119527.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_b85b1a95-35ba-473e-a164-4602a4a13a2f_2026-03-18_16:01:01.534496.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_0f235f9a-64f8-4790-917d-23e0a15077c5_2026-03-18_15:10:43.190962

...

/var/folders/ql/c79k160s5zjd1mzy_k08xvc40000gq/T/ipykernel_23082/2063899696.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")


/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_4b331fd2-8e92-4f74-90cd-615d90694463_2026-03-18_13:33:23.013609.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_945f0e58-761c-4c89-af96-0261821fcf11_2026-03-18_13:15:34.001010.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_f97dc224-8c40-4a1e-be11-d71eeca0c510_2026-03-18_13:40:07.411481.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_06908246-ff47-414c-9db2-77df83e43aee_2026-03-18_13:20:12.327352.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_379ba896-bb8f-440d-9d32-5dc9ca0297fe_2026-03-18_12:25:21.835522.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_64636904-aab6-4e10-a4dc-39638117731a_2026-03-18_11:17:53.999799.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_cfb7ac32-173c-4a66-9ee0-771a20eb93ed_2026-03-18_14:49:52.922613

/var/folders/ql/c79k160s5zjd1mzy_k08xvc40000gq/T/ipykernel_23082/2063899696.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")


/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_278f349f-1552-41c4-bec2-1de41838f229_2026-03-18_11:44:25.251416.pickle


/var/folders/ql/c79k160s5zjd1mzy_k08xvc40000gq/T/ipykernel_23082/2063899696.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")


And now we should be able to get an overall assessment...

In [229]:
outputs_ls[1]['results']

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
response_id,,,,,,,,,,,,,,,,,,,,,
r_0301,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0349,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0364,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0713,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_1842,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_1948,NaN,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_4050,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,incorrect,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_4627,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,incorrect,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_5119,NaN,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [230]:
outputs_ls[2]['results']

,0,1,2,3,4
response_id,,,,,
r_2525,NaN,correct,NaN,NaN,NaN
r_2583,NaN,NaN,NaN,NaN,NaN
r_3980,NaN,NaN,NaN,NaN,incorrect
r_6008,NaN,correct,NaN,NaN,NaN
r_7624,NaN,correct,NaN,NaN,NaN


First, get a big table of all the predictions:

In [231]:
predictions_df=pd.concat([o['results'] for o in outputs_ls])
predictions_df

,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
response_id,,,,,,,,,,,,,,,,,,,,,
r_0260,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2226,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2713,NaN,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_3728,incorrect,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_6429,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
r_5178,NaN,NaN,NaN,NaN,incorrect,NaN,NaN,correct,NaN,incorrect,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_5265,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_5328,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [232]:
# Let's define an ordered category for the prediction:

prediction_type = pd.CategoricalDtype(
    categories=['incorrect',  'correct'],
    ordered=True
)
prediction_type

CategoricalDtype(categories=['incorrect', 'correct'], ordered=True, categories_dtype=object)

And we can use this type on the predictions DataFrame:

In [233]:
predictions_df=predictions_df.astype(prediction_type)
predictions_df.head()

,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
response_id,,,,,,,,,,,,,,,,,,,,,
r_0260,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2226,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_2713,NaN,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_3728,incorrect,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_6429,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Now, an interesting comparison might be whether we choose the first prediction or the last prediction. ie. Do later predictions override the earlier ones, or should the first prediction stand?

In [234]:
predict_first_ss=predictions_df.bfill(axis='columns').iloc[:, 0].fillna('incorrect')
predict_first_ss

response_id
r_0260    incorrect
r_2226    incorrect
r_2713    incorrect
r_3728    incorrect
r_6429    incorrect
            ...    
r_5178    incorrect
r_5265    incorrect
r_5328    incorrect
r_6417    incorrect
r_6682    incorrect
Name: 0, Length: 827, dtype: category
Categories (2, object): ['incorrect' < 'correct']

In [235]:
predict_last_ss=predictions_df.ffill(axis='columns').iloc[:, -1].fillna('incorrect')
predict_last_ss

response_id
r_0260    incorrect
r_2226    incorrect
r_2713    incorrect
r_3728    incorrect
r_6429    incorrect
            ...    
r_5178    incorrect
r_5265    incorrect
r_5328    incorrect
r_6417    incorrect
r_6682    incorrect
Name: 33, Length: 827, dtype: category
Categories (2, object): ['incorrect' < 'correct']

And now we need to do a comparison with the ground truth results.

In [236]:
results_df = (
    theory["all_responses_df"]
    .query("`partition`=='trial'")
    .set_index("response_id")[["score"]]
    .rename({"score": "actual"}, axis="columns")
    .astype(prediction_type)
    .assign(predict_first=predict_first_ss, predict_last=predict_last_ss)
)

results_df

,actual,predict_first,predict_last
response_id,,,
r_1565,incorrect,incorrect,incorrect
r_2712,incorrect,correct,correct
r_2987,incorrect,incorrect,incorrect
r_3994,incorrect,incorrect,incorrect
r_4566,incorrect,incorrect,incorrect
...,...,...,...
r_7743,correct,incorrect,incorrect
r_1396,incorrect,incorrect,incorrect
r_4071,incorrect,incorrect,incorrect


In [237]:
results_df.query('`predict_first`!=`predict_last`')

,actual,predict_first,predict_last
response_id,,,
r_3735,correct,incorrect,correct
r_4164,correct,correct,incorrect
r_6282,incorrect,incorrect,correct
r_7104,correct,incorrect,correct
r_5923,correct,incorrect,correct
r_7703,incorrect,incorrect,correct
r_6980,incorrect,correct,incorrect
r_6306,incorrect,incorrect,correct
r_4058,incorrect,incorrect,correct


In [238]:
results_df.dropna()

,actual,predict_first,predict_last
response_id,,,
r_1565,incorrect,incorrect,incorrect
r_2712,incorrect,correct,correct
r_2987,incorrect,incorrect,incorrect
r_3994,incorrect,incorrect,incorrect
r_4566,incorrect,incorrect,incorrect
...,...,...,...
r_7743,correct,incorrect,incorrect
r_1396,incorrect,incorrect,incorrect
r_4071,incorrect,incorrect,incorrect


In [239]:
f1_score(results_df.dropna()['actual'], results_df.dropna()['predict_first'], average='weighted')

0.7802638597361653

In [240]:
f1_score(results_df.dropna()['actual'], results_df.dropna()['predict_last'], average='weighted')

0.7586604680256408

OK, predict first seems slightly better, as expected, although they're both pretty dire. Now let's do the evaluation with CWK:

In [241]:
cohen_kappa_score(results_df.dropna()['actual'], results_df.dropna()['predict_first'], weights='quadratic')

0.4527227061814777

That's hardly going to set the world alight...